> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
# ============================================================
# FIGURE 4 — MYELOID–STROMAL DRIVER DECOMPOSITION
# MAC / Mono × FIB / Fibrotic MC
# Formal 782-ROI + PC1 framework
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    exist_ok=True
)

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 1. Load data
# ============================================================

adata = sc.read_h5ad(
    big_path,
    backed="r"
)

roi_ad = sc.read_h5ad(
    roi_path
)

obs = adata.obs

print("Large dataset:", adata.shape)
print("ROI dataset:", roi_ad.shape)


# ============================================================
# 2. Reconstruct formal 782 ROI cells
# ============================================================

roi_cells = obs[
    obs["is_in_polygon"].to_numpy() == True
].copy()

roi_cells["roi_id"] = (
    roi_cells["polygon_flags"]
    .astype(str)
)

# 去掉expanded polygon重叠区域
roi_cells = roi_cells[
    ~roi_cells["roi_id"].str.contains(
        ",",
        regex=False
    )
].copy()

formal_roi_ids = set(
    roi_ad.obs_names.astype(str)
)

roi_cells = roi_cells[
    roi_cells["roi_id"].isin(
        formal_roi_ids
    )
].copy()

print("\nFormal ROI cells:")
print(len(roi_cells))

print("Unique ROI:")
print(
    roi_cells["roi_id"].nunique()
)


# ============================================================
# 3. Common PC1 support
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(
        ["min", "max"]
    )
)

pc_low = (
    pc_ranges["min"].max()
)

pc_high = (
    pc_ranges["max"].min()
)

print(
    "\nCommon PC1 support:",
    pc_low,
    "to",
    pc_high
)


# ============================================================
# 4. Common-support ROI metadata
# ============================================================

roi_common = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["ANCA", "SLE", "GBM"]
    )
    &
    roi_ad.obs["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()

common_roi_ids = set(
    roi_common.index.astype(str)
)

print("\nROI counts:")
print(
    roi_common["Disease"].value_counts()
)

print("\nPatient counts:")
print(
    roi_common
    .groupby(
        "Disease",
        observed=True
    )["Patient_Sample_ID"]
    .nunique()
)


# ============================================================
# 5. Restrict cell-level data
# ============================================================

spatial_cells = roi_cells[
    roi_cells["roi_id"].isin(
        common_roi_ids
    )
].copy()

print(
    "\nCells in common-support ROIs:",
    len(spatial_cells)
)


# ============================================================
# 6. Confirm the four cell types exist
# ============================================================

required_celltypes = [
    "MAC",
    "Mono",
    "FIB",
    "Fibrotic MC"
]

print(
    "\nRequired cell types:"
)

for ct in required_celltypes:

    n = (
        spatial_cells[
            "celltype_l1"
        ]
        .astype(str)
        .eq(ct)
        .sum()
    )

    print(
        ct,
        "=",
        n,
        "cells"
    )


# ============================================================
# 7. Neighbor enrichment function
#
# log2(
# observed target fraction among k nearest neighbors
# /
# expected target fraction within that ROI
# )
# ============================================================

def neighbor_enrichment(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3
):

    labels = (
        d["celltype_l1"]
        .astype(str)
        .to_numpy()
    )

    src_mask = (
        labels == source
    )

    tgt_mask = (
        labels == target
    )

    n = len(d)

    n_src = int(
        src_mask.sum()
    )

    n_tgt = int(
        tgt_mask.sum()
    )

    if (
        n < 5
        or n_src < min_source
        or n_tgt < min_target
    ):
        return np.nan

    xy = d[
        [
            "x_centroid",
            "y_centroid"
        ]
    ].to_numpy()

    tree = cKDTree(
        xy
    )

    kk = min(
        k + 1,
        n
    )

    _, nbr_idx = tree.query(
        xy[src_mask],
        k=kk
    )

    # 如果只有一个neighbor时，保证二维
    if nbr_idx.ndim == 1:
        nbr_idx = nbr_idx[:, None]

    # 第一列是source cell自己
    nbr_idx = nbr_idx[:, 1:]

    if nbr_idx.shape[1] == 0:
        return np.nan

    observed = (
        tgt_mask[
            nbr_idx
        ]
        .mean()
    )

    expected = (
        n_tgt /
        (n - 1)
    )

    eps = 1e-6

    return np.log2(
        (observed + eps)
        /
        (expected + eps)
    )


# ============================================================
# 8. Four pre-specified component relationships
# ============================================================

relations = [
    (
        "MAC",
        "FIB",
        "MAC_to_FIB"
    ),
    (
        "MAC",
        "Fibrotic MC",
        "MAC_to_FibroticMC"
    ),
    (
        "Mono",
        "FIB",
        "Mono_to_FIB"
    ),
    (
        "Mono",
        "Fibrotic MC",
        "Mono_to_FibroticMC"
    )
]


# ============================================================
# 9. Calculate k=6 enrichment for every ROI
# ============================================================

rows = []

for roi_id, d in spatial_cells.groupby(
    "roi_id",
    observed=True
):

    row = {
        "roi_id": str(
            roi_id
        )
    }

    for source, target, name in relations:

        row[name] = (
            neighbor_enrichment(
                d,
                source=source,
                target=target,
                k=6
            )
        )

    rows.append(
        row
    )


driver_df = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "roi_id"
    )
)


# ============================================================
# 10. Add metadata
# ============================================================

driver_df["Disease"] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Disease"
    ]
    .astype(str)
)

driver_df["Patient"] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Patient_Sample_ID"
    ]
    .astype(str)
)

driver_df["PC1"] = (
    roi_ad.obs.loc[
        driver_df.index,
        "PC1_crescent"
    ]
)


driver_vars = [
    "MAC_to_FIB",
    "MAC_to_FibroticMC",
    "Mono_to_FIB",
    "Mono_to_FibroticMC"
]


print(
    "\n================================"
)

print(
    "NON-MISSING ROIs"
)

print(
    "================================"
)


for var in driver_vars:

    print(
        "\n",
        var
    )

    print(
        driver_df
        .groupby(
            "Disease",
            observed=True
        )[var]
        .agg(
            total="size",
            nonmissing="count"
        )
    )


# ============================================================
# 11. Check independent patients retained
# ============================================================

print(
    "\n================================"
)

print(
    "PATIENTS RETAINED"
)

print(
    "================================"
)


for var in driver_vars:

    print(
        "\n",
        var
    )

    temp = (
        driver_df
        .dropna(
            subset=[var]
        )
    )

    print(
        temp
        .groupby(
            "Disease",
            observed=True
        )["Patient"]
        .nunique()
    )


# ============================================================
# 12. Formal SLE/LN vs GBM spline interaction tests
# ============================================================

test_rows = []


for var in driver_vars:

    d = driver_df[
        driver_df["Disease"].isin(
            ["SLE", "GBM"]
        )
    ].dropna(
        subset=[var]
    ).copy()


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # patient-balanced weighting
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform(
            "size"
        )
    )

    d["patient_weight"] = (
        1.0 /
        n_roi
    )


    fit = smf.wls(
        (
            f"{var} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
            d["Patient"]
        }
    )


    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    wt = fit.wald_test(
        R,
        scalar=True
    )


    test_rows.append(
        [
            var,
            float(
                wt.pvalue
            ),
            np.linalg.matrix_rank(
                fit.model.exog
            ),
            fit.model.exog.shape[1],
            d["Patient"].nunique(),
            d.loc[
                d["Disease"].astype(str)
                == "GBM",
                "Patient"
            ].nunique()
        ]
    )


driver_results = pd.DataFrame(
    test_rows,
    columns=[
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns",
        "n_patients_total",
        "n_GBM_patients"
    ]
)


# BH across the four pre-specified decomposed relationships
driver_results["FDR"] = multipletests(
    driver_results["pvalue"],
    method="fdr_bh"
)[1]


driver_results = (
    driver_results
    .sort_values(
        "FDR"
    )
)


print(
    "\n================================"
)

print(
    "LN vs anti-GBM DRIVER RESULTS"
)

print(
    "================================"
)

display(
    driver_results
)


# ============================================================
# 13. Rank check
# ============================================================

print(
    "\nNon-full-rank models:"
)

display(
    driver_results[
        driver_results["matrix_rank"]
        !=
        driver_results["n_columns"]
    ]
)


# ============================================================
# 14. Screening trajectory figure
# ============================================================

palette = {
    "SLE": "#E69F00",
    "GBM": "#009E73"
}

display_name = {
    "SLE": "LN",
    "GBM": "anti-GBM"
}


pretty_title = {
    "MAC_to_FIB":
        "MAC → FIB",

    "MAC_to_FibroticMC":
        "MAC → Fibrotic MC",

    "Mono_to_FIB":
        "Mono → FIB",

    "Mono_to_FibroticMC":
        "Mono → Fibrotic MC"
}


fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        12,
        9
    )
)

axes = (
    axes.flatten()
)


for ax, var in zip(
    axes,
    driver_vars
):

    for disease in [
        "SLE",
        "GBM"
    ]:

        d = driver_df[
            driver_df["Disease"]
            == disease
        ].dropna(
            subset=[var]
        ).sort_values(
            "PC1"
        )


        ax.scatter(
            d["PC1"],
            d[var],
            s=13,
            alpha=0.16,
            color=palette[
                disease
            ]
        )


        if len(d) >= 10:

            sm = lowess(
                d[var],
                d["PC1"],
                frac=0.55,
                return_sorted=True
            )


            ax.plot(
                sm[:, 0],
                sm[:, 1],
                linewidth=2.2,
                color=palette[
                    disease
                ],
                label=display_name[
                    disease
                ]
            )


    fdr = float(
        driver_results.loc[
            driver_results[
                "relationship"
            ]
            == var,
            "FDR"
        ].iloc[0]
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_title(
        (
            pretty_title[var]
            + "\n"
            + f"LN vs anti-GBM FDR={fdr:.3g}"
        ),
        fontsize=11
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )

    ax.set_ylabel(
        "log2 neighbor enrichment"
    )


axes[0].legend(
    frameon=False
)


plt.tight_layout(
    w_pad=2.5,
    h_pad=3
)


screen_path = (
    figdir /
    "Figure4_SCREEN_myeloid_stromal_decomposition.png"
)


plt.savefig(
    screen_path,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 15. Save data/results
# ============================================================

driver_df.to_csv(
    base /
    "figure4_driver_neighbor_data.csv"
)


driver_results.to_csv(
    base /
    "figure4_driver_results.csv",
    index=False
)


print(
    "\n================================"
)

print(
    "FIGURE 4 SCREEN COMPLETE"
)

print(
    "================================"
)

print(
    "\nSaved screening figure:"
)

print(
    screen_path
)

In [ ]:
# ============================================================
# FIGURE 4 — MEMORY-SAFE SETUP
# 只读取大h5ad中的obs metadata，不读取X_pca / UMAP / expression
# ============================================================

from pathlib import Path

import h5py
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"
figdir.mkdir(exist_ok=True)

big_path = base / "GSE294965_processed_data.h5ad"
roi_path = base / "roi_782_PC1_primary.h5ad"


# ============================================================
# 1. 只读取大文件的 obs
# ============================================================

# 兼容你现在的anndata版本
try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem


with h5py.File(big_path, "r") as f:
    obs = read_elem(f["obs"])


print("obs shape:", obs.shape)

print(
    [
        c for c in [
            "Disease",
            "Patient_Sample_ID",
            "celltype_l1",
            "is_in_polygon",
            "polygon_flags",
            "x_centroid",
            "y_centroid"
        ]
        if c in obs.columns
    ]
)


# ============================================================
# 2. 782 ROI小文件正常读取
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)

print("ROI dataset:", roi_ad.shape)

In [ ]:
# ============================================================
# ONE-TIME MEMORY-SAFE METADATA EXTRACTION
#
# 不读取：
# X / counts / PCA / UMAP / 整个obs
#
# 只分块读取Figure 4需要的字段
# ============================================================

from pathlib import Path

import h5py
import anndata as ad
import pandas as pd
import numpy as np


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)

slim_path = (
    base /
    "roi782_cell_metadata.pkl"
)


# ============================================================
# 1. Read the small 782-ROI file
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)

formal_roi_ids = set(
    roi_ad.obs_names.astype(str)
)

print(
    "Formal ROI IDs:",
    len(formal_roi_ids)
)


# ============================================================
# 2. Helper: decode byte strings
# ============================================================

def decode_array(arr):

    arr = np.asarray(arr)

    if arr.dtype.kind in {
        "S",
        "O",
        "U"
    }:

        return np.array([
            (
                x.decode("utf-8")
                if isinstance(
                    x,
                    (bytes, np.bytes_)
                )
                else str(x)
            )
            for x in arr
        ], dtype=object)

    return arr


# ============================================================
# 3. Helper:
# read ONE obs column for only ONE row slice
#
# Supports:
# normal datasets
# categoricals
# nullable arrays
# ============================================================

def read_obs_slice(
    obs_group,
    name,
    row_slice
):

    node = obs_group[name]

    # --------------------------
    # ordinary dataset
    # --------------------------

    if isinstance(
        node,
        h5py.Dataset
    ):

        return decode_array(
            node[row_slice]
        )


    # --------------------------
    # categorical column
    # --------------------------

    if (
        "codes" in node
        and
        "categories" in node
    ):

        codes = np.asarray(
            node["codes"][
                row_slice
            ]
        )

        categories = decode_array(
            node[
                "categories"
            ][:]
        )

        result = np.empty(
            len(codes),
            dtype=object
        )

        result[:] = None

        valid = (
            codes >= 0
        )

        result[
            valid
        ] = categories[
            codes[
                valid
            ]
        ]

        return result


    # --------------------------
    # nullable column
    # --------------------------

    if (
        "values" in node
        and
        "mask" in node
    ):

        values = decode_array(
            node[
                "values"
            ][row_slice]
        )

        mask = np.asarray(
            node[
                "mask"
            ][row_slice]
        )

        values = np.asarray(
            values,
            dtype=object
        )

        values[
            mask
        ] = None

        return values


    raise ValueError(
        f"Unsupported obs encoding: "
        f"{name}, keys={list(node.keys())}"
    )


# ============================================================
# 4. Helper:
# robust True/False conversion
# ============================================================

def to_bool_array(arr):

    arr = np.asarray(arr)

    if arr.dtype == bool:
        return arr

    if np.issubdtype(
        arr.dtype,
        np.number
    ):
        return (
            arr != 0
        )

    return np.fromiter(
        (
            str(x).lower()
            in {
                "true",
                "1",
                "t",
                "yes"
            }
            for x in arr
        ),
        dtype=bool,
        count=len(arr)
    )


# ============================================================
# 5. Chunk through 3.2 million cells
#
# 每次只处理100,000行
# ============================================================

chunk_size = 100_000

pieces = []


with h5py.File(
    big_path,
    "r"
) as f:

    og = f["obs"]

    # 用is_in_polygon确定总行数
    node = og[
        "is_in_polygon"
    ]

    if isinstance(
        node,
        h5py.Dataset
    ):

        n_total = (
            node.shape[0]
        )

    elif "codes" in node:

        n_total = (
            node[
                "codes"
            ].shape[0]
        )

    elif "values" in node:

        n_total = (
            node[
                "values"
            ].shape[0]
        )

    else:

        raise ValueError(
            "Cannot determine "
            "number of cells."
        )


    print(
        "Total cells:",
        n_total
    )


    for start in range(
        0,
        n_total,
        chunk_size
    ):

        stop = min(
            start + chunk_size,
            n_total
        )

        sl = slice(
            start,
            stop
        )


        # --------------------------------
        # First filter:
        # cells inside expanded polygons
        # --------------------------------

        in_poly = to_bool_array(
            read_obs_slice(
                og,
                "is_in_polygon",
                sl
            )
        )


        if not in_poly.any():
            continue


        # --------------------------------
        # Read polygon ID
        # --------------------------------

        poly = read_obs_slice(
            og,
            "polygon_flags",
            sl
        )


        # --------------------------------
        # Second filter:
        # only the formal 782 ROI IDs
        # --------------------------------

        formal_mask = np.fromiter(
            (
                (
                    bool(in_poly[i])
                    and
                    str(poly[i])
                    in formal_roi_ids
                )
                for i in range(
                    len(poly)
                )
            ),
            dtype=bool,
            count=len(poly)
        )


        if not formal_mask.any():
            continue


        # --------------------------------
        # Read ONLY required columns
        # --------------------------------

        disease = read_obs_slice(
            og,
            "Disease",
            sl
        )

        patient = read_obs_slice(
            og,
            "Patient_Sample_ID",
            sl
        )

        celltype = read_obs_slice(
            og,
            "celltype_l1",
            sl
        )

        x = read_obs_slice(
            og,
            "x_centroid",
            sl
        )

        y = read_obs_slice(
            og,
            "y_centroid",
            sl
        )


        piece = pd.DataFrame({
            "roi_id":
                np.asarray(poly)[
                    formal_mask
                ],

            "Disease":
                np.asarray(disease)[
                    formal_mask
                ],

            "Patient":
                np.asarray(patient)[
                    formal_mask
                ],

            "celltype_l1":
                np.asarray(celltype)[
                    formal_mask
                ],

            "x_centroid":
                np.asarray(x)[
                    formal_mask
                ].astype(float),

            "y_centroid":
                np.asarray(y)[
                    formal_mask
                ].astype(float)
        })


        pieces.append(
            piece
        )


        print(
            f"{stop:,} / "
            f"{n_total:,}"
        )


# ============================================================
# 6. Combine the small pieces
# ============================================================

roi_cells_meta = pd.concat(
    pieces,
    ignore_index=True
)

del pieces


# ============================================================
# 7. Reduce RAM usage
# ============================================================

for col in [
    "roi_id",
    "Disease",
    "Patient",
    "celltype_l1"
]:

    roi_cells_meta[
        col
    ] = roi_cells_meta[
        col
    ].astype(
        "category"
    )


print(
    "\nFinal metadata shape:",
    roi_cells_meta.shape
)

print(
    "Unique ROI:",
    roi_cells_meta[
        "roi_id"
    ].nunique()
)

print(
    "\nDiseases:"
)

print(
    roi_cells_meta[
        "Disease"
    ].value_counts()
)

print(
    "\nMemory usage MB:"
)

print(
    roi_cells_meta
    .memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


# ============================================================
# 8. SAVE
#
# Save compact ROI-level objects for downstream analyses.
# Subsequent steps can use these files without reopening the full 3.2-million-cell h5ad.
# ============================================================

roi_cells_meta.to_pickle(
    slim_path
)

print(
    "\nSaved:"
)

print(
    slim_path
)

In [ ]:
# ============================================================
# FIGURE 4 — DRIVER DECOMPOSITION
#
# MAC / Mono
# ×
# FIB / Fibrotic MC
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 1. Read lightweight metadata
# ============================================================

roi_cells_meta = pd.read_pickle(
    base /
    "roi782_cell_metadata.pkl"
)

print(
    "Cell metadata:",
    roi_cells_meta.shape
)


# ============================================================
# 2. Common PC1 support
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs[
            "Disease"
        ].isin(
            [
                "ANCA",
                "SLE",
                "GBM"
            ]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        ["min", "max"]
    )
)

pc_low = (
    pc_ranges[
        "min"
    ].max()
)

pc_high = (
    pc_ranges[
        "max"
    ].min()
)


roi_common = roi_ad.obs[
    roi_ad.obs[
        "Disease"
    ].isin(
        [
            "ANCA",
            "SLE",
            "GBM"
        ]
    )
    &
    roi_ad.obs[
        "PC1_crescent"
    ].between(
        pc_low,
        pc_high
    )
].copy()


common_ids = set(
    roi_common
    .index
    .astype(str)
)


# ============================================================
# 3. Keep common-support cells
# ============================================================

spatial_cells = (
    roi_cells_meta[
        roi_cells_meta[
            "roi_id"
        ]
        .astype(str)
        .isin(
            common_ids
        )
    ]
    .copy()
)


print(
    "\nCommon-support cells:",
    len(spatial_cells)
)


# ============================================================
# 4. Check four required cell types
# ============================================================

required = [
    "MAC",
    "Mono",
    "FIB",
    "Fibrotic MC"
]


print(
    "\nCell counts:"
)


for ct in required:

    n = (
        spatial_cells[
            "celltype_l1"
        ]
        .astype(str)
        .eq(ct)
        .sum()
    )

    print(
        ct,
        n
    )


# ============================================================
# 5. Neighbor enrichment function
# ============================================================

def neighbor_enrichment(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3
):

    labels = (
        d[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
    )

    src = (
        labels == source
    )

    tgt = (
        labels == target
    )

    n = len(d)

    if (
        src.sum()
        < min_source
        or
        tgt.sum()
        < min_target
        or
        n < 5
    ):

        return np.nan


    xy = d[
        [
            "x_centroid",
            "y_centroid"
        ]
    ].to_numpy()


    tree = cKDTree(
        xy
    )


    kk = min(
        k + 1,
        n
    )


    _, nbr = tree.query(
        xy[src],
        k=kk
    )


    if nbr.ndim == 1:

        nbr = (
            nbr[:, None]
        )


    nbr = nbr[:, 1:]


    if nbr.shape[1] == 0:

        return np.nan


    observed = (
        tgt[
            nbr
        ]
        .mean()
    )


    expected = (
        tgt.sum()
        /
        (n - 1)
    )


    eps = 1e-6


    return np.log2(
        (
            observed + eps
        )
        /
        (
            expected + eps
        )
    )


# ============================================================
# 6. Four pre-specified relationships
# ============================================================

relations = [
    (
        "MAC",
        "FIB",
        "MAC_to_FIB"
    ),

    (
        "MAC",
        "Fibrotic MC",
        "MAC_to_FibroticMC"
    ),

    (
        "Mono",
        "FIB",
        "Mono_to_FIB"
    ),

    (
        "Mono",
        "Fibrotic MC",
        "Mono_to_FibroticMC"
    )
]


# ============================================================
# 7. Calculate enrichment
# ============================================================

rows = []


for roi_id, d in spatial_cells.groupby(
    "roi_id",
    observed=True
):

    row = {
        "roi_id":
            str(roi_id)
    }


    for source, target, name in relations:

        row[name] = (
            neighbor_enrichment(
                d,
                source,
                target,
                k=6
            )
        )


    rows.append(
        row
    )


driver_df = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "roi_id"
    )
)


# ============================================================
# 8. Add metadata
# ============================================================

driver_df[
    "Disease"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Disease"
    ]
    .astype(str)
)


driver_df[
    "Patient"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Patient_Sample_ID"
    ]
    .astype(str)
)


driver_df[
    "PC1"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "PC1_crescent"
    ]
)


driver_vars = [
    "MAC_to_FIB",
    "MAC_to_FibroticMC",
    "Mono_to_FIB",
    "Mono_to_FibroticMC"
]


# ============================================================
# 9. Coverage checks
# ============================================================

print(
    "\n=============================="
)

print(
    "NON-MISSING ROIs"
)

print(
    "=============================="
)


for var in driver_vars:

    print(
        "\n",
        var
    )

    print(
        driver_df
        .groupby(
            "Disease",
            observed=True
        )[var]
        .agg(
            total="size",
            nonmissing="count"
        )
    )


print(
    "\n=============================="
)

print(
    "PATIENTS RETAINED"
)

print(
    "=============================="
)


for var in driver_vars:

    d = (
        driver_df
        .dropna(
            subset=[var]
        )
    )

    print(
        "\n",
        var
    )

    print(
        d.groupby(
            "Disease",
            observed=True
        )[
            "Patient"
        ]
        .nunique()
    )


# ============================================================
# 10. Formal LN vs anti-GBM tests
# ============================================================

result_rows = []


for var in driver_vars:

    d = driver_df[
        driver_df[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ].dropna(
        subset=[var]
    ).copy()


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    fit = smf.wls(
        (
            f"{var} ~ "
            "bs(PC1, df=3, "
            "degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and
        "C(Disease)"
        in term
        and
        "bs(PC1"
        in term
    ]


    R = np.zeros(
        (
            len(terms),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        terms
    ):

        R[
            i,
            fit.params
            .index
            .get_loc(
                term
            )
        ] = 1


    wt = fit.wald_test(
        R,
        scalar=True
    )


    result_rows.append(
        [
            var,

            float(
                wt.pvalue
            ),

            np.linalg.matrix_rank(
                fit.model.exog
            ),

            fit.model.exog.shape[
                1
            ],

            d[
                "Patient"
            ].nunique(),

            d.loc[
                d[
                    "Disease"
                ]
                .astype(str)
                == "GBM",
                "Patient"
            ].nunique()
        ]
    )


driver_results = pd.DataFrame(
    result_rows,
    columns=[
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns",
        "n_patients",
        "n_GBM_patients"
    ]
)


driver_results[
    "FDR"
] = multipletests(
    driver_results[
        "pvalue"
    ],
    method="fdr_bh"
)[1]


driver_results = (
    driver_results
    .sort_values(
        "FDR"
    )
)


print(
    "\n=============================="
)

print(
    "LN vs anti-GBM RESULTS"
)

print(
    "=============================="
)


display(
    driver_results
)


print(
    "\nNon-full-rank:"
)


display(
    driver_results[
        driver_results[
            "matrix_rank"
        ]
        !=
        driver_results[
            "n_columns"
        ]
    ]
)


# ============================================================
# 11. Screening plot
# ============================================================

palette = {
    "SLE":
        "#E69F00",

    "GBM":
        "#009E73"
}


names = {
    "SLE":
        "LN",

    "GBM":
        "anti-GBM"
}


pretty = {
    "MAC_to_FIB":
        "MAC → FIB",

    "MAC_to_FibroticMC":
        "MAC → Fibrotic MC",

    "Mono_to_FIB":
        "Mono → FIB",

    "Mono_to_FibroticMC":
        "Mono → Fibrotic MC"
}


fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        12,
        9
    )
)


axes = axes.flatten()


for ax, var in zip(
    axes,
    driver_vars
):

    for disease in [
        "SLE",
        "GBM"
    ]:

        d = driver_df[
            driver_df[
                "Disease"
            ]
            == disease
        ].dropna(
            subset=[
                var
            ]
        ).sort_values(
            "PC1"
        )


        ax.scatter(
            d[
                "PC1"
            ],
            d[
                var
            ],
            s=13,
            alpha=0.16,
            color=palette[
                disease
            ]
        )


        if len(d) >= 10:

            sm = lowess(
                d[
                    var
                ],
                d[
                    "PC1"
                ],
                frac=0.55,
                return_sorted=True
            )


            ax.plot(
                sm[:, 0],
                sm[:, 1],
                linewidth=2.2,
                color=palette[
                    disease
                ],
                label=names[
                    disease
                ]
            )


    q = float(
        driver_results.loc[
            driver_results[
                "relationship"
            ]
            == var,
            "FDR"
        ].iloc[0]
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_title(
        pretty[var]
        + "\n"
        + f"LN vs anti-GBM FDR={q:.3g}"
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "log2 neighbor enrichment"
    )


axes[0].legend(
    frameon=False
)


plt.tight_layout()


screen_path = (
    figdir /
    "Figure4_SCREEN_driver_decomposition.png"
)


plt.savefig(
    screen_path,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 12. Save
# ============================================================

driver_df.to_csv(
    base /
    "figure4_driver_neighbor_data.csv"
)


driver_results.to_csv(
    base /
    "figure4_driver_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    screen_path
)

In [ ]:
from pathlib import Path

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)

print("Figure folder:")
print(figdir)

In [ ]:
# ============================================================
# FIGURE 4 — DRIVER DECOMPOSITION
#
# MAC / Mono
# ×
# FIB / Fibrotic MC
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 1. Read lightweight metadata
# ============================================================

roi_cells_meta = pd.read_pickle(
    base /
    "roi782_cell_metadata.pkl"
)

print(
    "Cell metadata:",
    roi_cells_meta.shape
)


# ============================================================
# 2. Common PC1 support
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs[
            "Disease"
        ].isin(
            [
                "ANCA",
                "SLE",
                "GBM"
            ]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        ["min", "max"]
    )
)

pc_low = (
    pc_ranges[
        "min"
    ].max()
)

pc_high = (
    pc_ranges[
        "max"
    ].min()
)


roi_common = roi_ad.obs[
    roi_ad.obs[
        "Disease"
    ].isin(
        [
            "ANCA",
            "SLE",
            "GBM"
        ]
    )
    &
    roi_ad.obs[
        "PC1_crescent"
    ].between(
        pc_low,
        pc_high
    )
].copy()


common_ids = set(
    roi_common
    .index
    .astype(str)
)


# ============================================================
# 3. Keep common-support cells
# ============================================================

spatial_cells = (
    roi_cells_meta[
        roi_cells_meta[
            "roi_id"
        ]
        .astype(str)
        .isin(
            common_ids
        )
    ]
    .copy()
)


print(
    "\nCommon-support cells:",
    len(spatial_cells)
)


# ============================================================
# 4. Check four required cell types
# ============================================================

required = [
    "MAC",
    "Mono",
    "FIB",
    "Fibrotic MC"
]


print(
    "\nCell counts:"
)


for ct in required:

    n = (
        spatial_cells[
            "celltype_l1"
        ]
        .astype(str)
        .eq(ct)
        .sum()
    )

    print(
        ct,
        n
    )


# ============================================================
# 5. Neighbor enrichment function
# ============================================================

def neighbor_enrichment(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3
):

    labels = (
        d[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
    )

    src = (
        labels == source
    )

    tgt = (
        labels == target
    )

    n = len(d)

    if (
        src.sum()
        < min_source
        or
        tgt.sum()
        < min_target
        or
        n < 5
    ):

        return np.nan


    xy = d[
        [
            "x_centroid",
            "y_centroid"
        ]
    ].to_numpy()


    tree = cKDTree(
        xy
    )


    kk = min(
        k + 1,
        n
    )


    _, nbr = tree.query(
        xy[src],
        k=kk
    )


    if nbr.ndim == 1:

        nbr = (
            nbr[:, None]
        )


    nbr = nbr[:, 1:]


    if nbr.shape[1] == 0:

        return np.nan


    observed = (
        tgt[
            nbr
        ]
        .mean()
    )


    expected = (
        tgt.sum()
        /
        (n - 1)
    )


    eps = 1e-6


    return np.log2(
        (
            observed + eps
        )
        /
        (
            expected + eps
        )
    )


# ============================================================
# 6. Four pre-specified relationships
# ============================================================

relations = [
    (
        "MAC",
        "FIB",
        "MAC_to_FIB"
    ),

    (
        "MAC",
        "Fibrotic MC",
        "MAC_to_FibroticMC"
    ),

    (
        "Mono",
        "FIB",
        "Mono_to_FIB"
    ),

    (
        "Mono",
        "Fibrotic MC",
        "Mono_to_FibroticMC"
    )
]


# ============================================================
# 7. Calculate enrichment
# ============================================================

rows = []


for roi_id, d in spatial_cells.groupby(
    "roi_id",
    observed=True
):

    row = {
        "roi_id":
            str(roi_id)
    }


    for source, target, name in relations:

        row[name] = (
            neighbor_enrichment(
                d,
                source,
                target,
                k=6
            )
        )


    rows.append(
        row
    )


driver_df = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "roi_id"
    )
)


# ============================================================
# 8. Add metadata
# ============================================================

driver_df[
    "Disease"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Disease"
    ]
    .astype(str)
)


driver_df[
    "Patient"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "Patient_Sample_ID"
    ]
    .astype(str)
)


driver_df[
    "PC1"
] = (
    roi_ad.obs.loc[
        driver_df.index,
        "PC1_crescent"
    ]
)


driver_vars = [
    "MAC_to_FIB",
    "MAC_to_FibroticMC",
    "Mono_to_FIB",
    "Mono_to_FibroticMC"
]


# ============================================================
# 9. Coverage checks
# ============================================================

print(
    "\n=============================="
)

print(
    "NON-MISSING ROIs"
)

print(
    "=============================="
)


for var in driver_vars:

    print(
        "\n",
        var
    )

    print(
        driver_df
        .groupby(
            "Disease",
            observed=True
        )[var]
        .agg(
            total="size",
            nonmissing="count"
        )
    )


print(
    "\n=============================="
)

print(
    "PATIENTS RETAINED"
)

print(
    "=============================="
)


for var in driver_vars:

    d = (
        driver_df
        .dropna(
            subset=[var]
        )
    )

    print(
        "\n",
        var
    )

    print(
        d.groupby(
            "Disease",
            observed=True
        )[
            "Patient"
        ]
        .nunique()
    )


# ============================================================
# 10. Formal LN vs anti-GBM tests
# ============================================================

result_rows = []


for var in driver_vars:

    d = driver_df[
        driver_df[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ].dropna(
        subset=[var]
    ).copy()


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    fit = smf.wls(
        (
            f"{var} ~ "
            "bs(PC1, df=3, "
            "degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and
        "C(Disease)"
        in term
        and
        "bs(PC1"
        in term
    ]


    R = np.zeros(
        (
            len(terms),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        terms
    ):

        R[
            i,
            fit.params
            .index
            .get_loc(
                term
            )
        ] = 1


    wt = fit.wald_test(
        R,
        scalar=True
    )


    result_rows.append(
        [
            var,

            float(
                wt.pvalue
            ),

            np.linalg.matrix_rank(
                fit.model.exog
            ),

            fit.model.exog.shape[
                1
            ],

            d[
                "Patient"
            ].nunique(),

            d.loc[
                d[
                    "Disease"
                ]
                .astype(str)
                == "GBM",
                "Patient"
            ].nunique()
        ]
    )


driver_results = pd.DataFrame(
    result_rows,
    columns=[
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns",
        "n_patients",
        "n_GBM_patients"
    ]
)


driver_results[
    "FDR"
] = multipletests(
    driver_results[
        "pvalue"
    ],
    method="fdr_bh"
)[1]


driver_results = (
    driver_results
    .sort_values(
        "FDR"
    )
)


print(
    "\n=============================="
)

print(
    "LN vs anti-GBM RESULTS"
)

print(
    "=============================="
)


display(
    driver_results
)


print(
    "\nNon-full-rank:"
)


display(
    driver_results[
        driver_results[
            "matrix_rank"
        ]
        !=
        driver_results[
            "n_columns"
        ]
    ]
)


# ============================================================
# 11. Screening plot
# ============================================================

palette = {
    "SLE":
        "#E69F00",

    "GBM":
        "#009E73"
}


names = {
    "SLE":
        "LN",

    "GBM":
        "anti-GBM"
}


pretty = {
    "MAC_to_FIB":
        "MAC → FIB",

    "MAC_to_FibroticMC":
        "MAC → Fibrotic MC",

    "Mono_to_FIB":
        "Mono → FIB",

    "Mono_to_FibroticMC":
        "Mono → Fibrotic MC"
}


fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        12,
        9
    )
)


axes = axes.flatten()


for ax, var in zip(
    axes,
    driver_vars
):

    for disease in [
        "SLE",
        "GBM"
    ]:

        d = driver_df[
            driver_df[
                "Disease"
            ]
            == disease
        ].dropna(
            subset=[
                var
            ]
        ).sort_values(
            "PC1"
        )


        ax.scatter(
            d[
                "PC1"
            ],
            d[
                var
            ],
            s=13,
            alpha=0.16,
            color=palette[
                disease
            ]
        )


        if len(d) >= 10:

            sm = lowess(
                d[
                    var
                ],
                d[
                    "PC1"
                ],
                frac=0.55,
                return_sorted=True
            )


            ax.plot(
                sm[:, 0],
                sm[:, 1],
                linewidth=2.2,
                color=palette[
                    disease
                ],
                label=names[
                    disease
                ]
            )


    q = float(
        driver_results.loc[
            driver_results[
                "relationship"
            ]
            == var,
            "FDR"
        ].iloc[0]
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_title(
        pretty[var]
        + "\n"
        + f"LN vs anti-GBM FDR={q:.3g}"
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "log2 neighbor enrichment"
    )


axes[0].legend(
    frameon=False
)


plt.tight_layout()


screen_path = (
    figdir /
    "Figure4_SCREEN_driver_decomposition.png"
)


plt.savefig(
    screen_path,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 12. Save
# ============================================================

driver_df.to_csv(
    base /
    "figure4_driver_neighbor_data.csv"
)


driver_results.to_csv(
    base /
    "figure4_driver_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    screen_path
)

In [ ]:
# ==== Cell 1: imports + output folder ====

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# LOWESS 平滑
from statsmodels.nonparametric.smoothers_lowess import lowess

# Figure style
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# Output directory (edit this path if needed)
figdir = Path(str(PROJECT_DIR / 'figures'))
figdir.mkdir(parents=True, exist_ok=True)

print("输出目录：", figdir)

In [ ]:
# ==== Cell 2: find existing objects + basic checks ====

def get_first_existing_object(candidate_names):
    """
    Search the current notebook environment for the first available object name.
    """
    for name in candidate_names:
        if name in globals():
            print(f"找到对象: {name}")
            return globals()[name], name
    return None, None


def find_first_existing_column(df, candidate_cols, must_exist=True):
    """
    Find the first matching column name in a dataframe.
    """
    for c in candidate_cols:
        if c in df.columns:
            return c
    if must_exist:
        raise ValueError(f"没找到这些候选列中的任何一个: {candidate_cols}")
    return None


def pretty_fdr(x):
    """
    美化 FDR 显示
    """
    if pd.isna(x):
        return "NA"
    if x < 1e-3:
        return f"{x:.2e}"
    return f"{x:.3f}"


def get_fdr_from_results(results_df, relation_name):
    """
    从已有结果表中提取某个 relationship 在 SLE_vs_GBM / LN_vs_GBM 对比中的 FDR
    The function is intentionally flexible to accommodate alternative column naming conventions.
    """
    if results_df is None:
        return np.nan

    rel_col = None
    for c in ["relationship", "neighbor_relation", "relation"]:
        if c in results_df.columns:
            rel_col = c
            break

    fdr_col = None
    for c in ["FDR", "fdr", "adj_p", "adj.P.Val"]:
        if c in results_df.columns:
            fdr_col = c
            break

    comp_col = None
    for c in ["comparison", "contrast", "group"]:
        if c in results_df.columns:
            comp_col = c
            break

    if rel_col is None or fdr_col is None:
        return np.nan

    tmp = results_df.copy()
    tmp = tmp[tmp[rel_col].astype(str) == relation_name].copy()

    # 如果有 comparison 列，则优先抓 SLE vs GBM / LN vs GBM
    if comp_col is not None and len(tmp) > 0:
        comp_series = tmp[comp_col].astype(str)
        mask1 = comp_series.str.contains("SLE", case=False, na=False) & comp_series.str.contains("GBM", case=False, na=False)
        mask2 = comp_series.str.contains("LN", case=False, na=False) & comp_series.str.contains("GBM", case=False, na=False)
        mask3 = comp_series.str.contains("anti", case=False, na=False) & comp_series.str.contains("LN", case=False, na=False)

        if mask1.any():
            tmp = tmp[mask1].copy()
        elif mask2.any():
            tmp = tmp[mask2].copy()
        elif mask3.any():
            tmp = tmp[mask3].copy()

    if len(tmp) == 0:
        return np.nan

    return tmp[fdr_col].iloc[0]


# ---- 1) 找主数据表 ----
neighbor_df, neighbor_name = get_first_existing_object([
    "pc1_neighbor",
    "neighbor_df",
    "neighbor_data",
    "pc1_neighbor_df"
])

if neighbor_df is None:
    raise ValueError(
        "Primary plotting data object not found. Ensure that an object such as pc1_neighbor or neighbor_df is available before running this cell."
    )

# ---- 2) Locate results table for FDR annotation ----
results_df, results_name = get_first_existing_object([
    "ln_gbm_results",
    "pc1_pairwise",
    "neighbor_pairwise",
    "pairwise_results",
    "spatial_pairwise"
])

if results_df is None:
    print("print("Pairwise results table not found. The figure can still be generated, but the FDR annotation may be unavailable.")

# ---- 3) 主数据列自动识别 ----
pc1_col = find_first_existing_column(
    neighbor_df,
    ["PC1", "pc1", "common_pc1", "crescent_pc1", "trajectory", "pseudotime"]
)

disease_col = find_first_existing_column(
    neighbor_df,
    ["Disease", "disease"]
)

patient_col = find_first_existing_column(
    neighbor_df,
    ["Patient_Sample_ID", "patient", "Patient", "sample", "Sample", "sample_id"],
    must_exist=False
)

# Primary spatial relationships displayed in the final Figure 4.
relation_candidates = ["MAC_to_FIB", "MAC_to_FibroticMC"]
missing_rel = [r for r in relation_candidates if r not in neighbor_df.columns]

if len(missing_rel) > 0:
    raise ValueError(f"Required relationship columns are missing: {missing_rel}
Check the preceding neighbor_relationship generation step.")

print("\n主数据对象：", neighbor_name)
print("PC1列：", pc1_col)
print("Disease列：", disease_col)
print("Patient列：", patient_col if patient_col is not None else "未找到（不影响作图）")

print("\n主数据维度：", neighbor_df.shape)
print("\n可用列预览：")
print(neighbor_df.columns.tolist()[:30])

if results_df is not None:
    print("\n结果表对象：", results_name)
    print("结果表维度：", results_df.shape)
    print("结果表列：", results_df.columns.tolist())

In [ ]:
# ============================================================
# FIGURE 4 — FINAL, SELF-CONTAINED VERSION
#
# Main findings:
#   MAC → FIB
#   MAC → Fibrotic MC
#
# 数据直接从硬盘读取
# 不依赖任何前面Notebook的内存变量
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


data_path = (
    base /
    "figure4_driver_neighbor_data.csv"
)

results_path = (
    base /
    "figure4_driver_results.csv"
)


print("读取：")
print(data_path)
print(results_path)


# ============================================================
# 1. Read Figure 4 data
# ============================================================

driver_df = pd.read_csv(
    data_path,
    index_col=0
)

driver_results = pd.read_csv(
    results_path
)


print("\nDriver data shape:")
print(driver_df.shape)

print("\nDriver data columns:")
print(driver_df.columns.tolist())

print("\nDriver results:")
display(driver_results)


# ============================================================
# 2. Check required columns
# ============================================================

required_columns = [
    "Disease",
    "Patient",
    "PC1",
    "MAC_to_FIB",
    "MAC_to_FibroticMC"
]

missing = [
    x for x in required_columns
    if x not in driver_df.columns
]

if len(missing) > 0:

    raise ValueError(
        "缺少这些列："
        + str(missing)
    )


print(
    "\n需要的列全部存在 ✓"
)


# ============================================================
# 3. Only LN(SLE) vs anti-GBM
# ============================================================

plot_df = driver_df[
    driver_df["Disease"].isin(
        ["SLE", "GBM"]
    )
].copy()


print(
    "\nROI counts:"
)

print(
    plot_df["Disease"]
    .value_counts()
)


print(
    "\nPatient counts:"
)

print(
    plot_df.groupby(
        "Disease",
        observed=True
    )["Patient"].nunique()
)


# ============================================================
# 4. Formal display names / colors
# ============================================================

display_name = {
    "SLE": "LN",
    "GBM": "anti-GBM"
}

palette = {
    "SLE": "#E69F00",
    "GBM": "#009E73"
}


relations = [
    "MAC_to_FIB",
    "MAC_to_FibroticMC"
]


pretty_name = {
    "MAC_to_FIB":
        "MAC → FIB",

    "MAC_to_FibroticMC":
        "MAC → Fibrotic MC"
}


panel_letter = {
    "MAC_to_FIB": "A",
    "MAC_to_FibroticMC": "B"
}


# ============================================================
# 5. FDR helper
# ============================================================

def get_fdr(
    relationship
):

    temp = driver_results[
        driver_results[
            "relationship"
        ]
        == relationship
    ]

    if len(temp) == 0:
        return np.nan

    return float(
        temp["FDR"].iloc[0]
    )


def format_fdr(x):

    if pd.isna(x):
        return "NA"

    if x < 0.001:
        return f"{x:.2e}"

    return f"{x:.3f}"


# ============================================================
# 6. Formal spline model function
#
# 和前面正式统计保持一致：
#
# y ~ spline(PC1) * Disease
#
# patient-balanced weights
# patient-clustered covariance
# ============================================================

def fit_and_predict(
    data,
    outcome
):

    d = data.dropna(
        subset=[
            outcome,
            "PC1",
            "Disease",
            "Patient"
        ]
    ).copy()


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # --------------------------
    # patient-balanced weighting
    # --------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform("size")
    )


    d["patient_weight"] = (
        1.0 /
        n_roi
    )


    # --------------------------
    # formal model
    # --------------------------

    fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d["Patient"]
        }
    )


    # --------------------------
    # common prediction grid
    # --------------------------

    x_min = (
        d["PC1"].min()
    )

    x_max = (
        d["PC1"].max()
    )


    grid = np.linspace(
        x_min,
        x_max,
        180
    )


    pred_list = []


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1": grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    * len(grid),

                    categories=[
                        "SLE",
                        "GBM"
                    ]
                )
        })


        pred = (
            fit
            .get_prediction(
                newdata
            )
            .summary_frame(
                alpha=0.05
            )
        )


        temp = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                disease,

            "fit":
                pred[
                    "mean"
                ].to_numpy(),

            "lower":
                pred[
                    "mean_ci_lower"
                ].to_numpy(),

            "upper":
                pred[
                    "mean_ci_upper"
                ].to_numpy()
        })


        pred_list.append(
            temp
        )


    pred_df = pd.concat(
        pred_list,
        ignore_index=True
    )


    return (
        d,
        fit,
        pred_df
    )


# ============================================================
# 7. Create Figure 4
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.8, 5.3)
)


source_tables = []


for ax, relation in zip(
    axes,
    relations
):

    # --------------------------
    # formal fit
    # --------------------------

    d, fit, pred_df = (
        fit_and_predict(
            plot_df,
            relation
        )
    )


    # --------------------------
    # IMPORTANT:
    # 统计模型使用原始数据。
    #
    # 这里只对散点显示进行1%-99%裁剪，
    # 防止 -15 之类极端值把整张图压扁。
    #
    # 不改变统计结果。
    # --------------------------

    q01 = d[
        relation
    ].quantile(
        0.01
    )

    q99 = d[
        relation
    ].quantile(
        0.99
    )


    d[
        "display_y"
    ] = (
        d[relation]
        .clip(
            lower=q01,
            upper=q99
        )
    )


    # --------------------------
    # Scatter + formal model
    # --------------------------

    for disease in [
        "SLE",
        "GBM"
    ]:

        raw = d[
            d["Disease"]
            .astype(str)
            == disease
        ]


        pred = pred_df[
            pred_df["Disease"]
            == disease
        ]


        color = (
            palette[
                disease
            ]
        )


        # ROI scatter
        ax.scatter(
            raw["PC1"],
            raw["display_y"],
            s=17,
            alpha=0.20,
            color=color
        )


        # fitted trajectory
        ax.plot(
            pred["PC1"],
            pred["fit"],
            linewidth=2.6,
            color=color,
            label=display_name[
                disease
            ]
        )


        # 95% confidence interval
        ax.fill_between(
            pred["PC1"],
            pred["lower"],
            pred["upper"],
            color=color,
            alpha=0.15
        )


    # --------------------------
    # Random-mixing reference
    # --------------------------

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="#4C78A8"
    )


    # --------------------------
    # FDR
    # --------------------------

    fdr = get_fdr(
        relation
    )


    ax.set_title(
        (
            f"{panel_letter[relation]}  "
            f"{pretty_name[relation]}"
            "\n"
            "LN vs anti-GBM "
            f"interaction FDR = "
            f"{format_fdr(fdr)}"
        ),
        loc="left",
        fontsize=12
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "log2 neighbor enrichment"
    )


    ax.legend(
        frameon=False
    )


    # --------------------------
    # Clean figure appearance
    # --------------------------

    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


    # --------------------------
    # Sensible Y range
    #
    # combine:
    # clipped raw points
    # + model CI
    # --------------------------

    y_candidates = np.concatenate(
        [
            d[
                "display_y"
            ].to_numpy(),

            pred_df[
                "lower"
            ].to_numpy(),

            pred_df[
                "upper"
            ].to_numpy()
        ]
    )


    y_candidates = (
        y_candidates[
            np.isfinite(
                y_candidates
            )
        ]
    )


    ymin = (
        np.min(
            y_candidates
        )
    )

    ymax = (
        np.max(
            y_candidates
        )
    )


    yrange = (
        ymax - ymin
    )


    if yrange <= 0:
        yrange = 1


    ax.set_ylim(
        ymin
        - 0.08
        * yrange,

        ymax
        + 0.08
        * yrange
    )


    # --------------------------
    # Save source data
    # --------------------------

    export = d[
        [
            "Disease",
            "Patient",
            "PC1",
            relation
        ]
    ].copy()


    export[
        "relationship"
    ] = relation


    source_tables.append(
        export
    )


# ============================================================
# 8. Final layout
# ============================================================

plt.tight_layout(
    w_pad=3.0
)


# ============================================================
# 9. Save Figure
# ============================================================

png_path = (
    figdir /
    "Figure4_FINAL_MAC_stromal_decomposition.png"
)

pdf_path = (
    figdir /
    "Figure4_FINAL_MAC_stromal_decomposition.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 10. Save source data
# ============================================================

source_df = pd.concat(
    source_tables,
    ignore_index=True
)


source_path = (
    figdir /
    "Figure4_FINAL_source_data.csv"
)


source_df.to_csv(
    source_path,
    index=False
)


# ============================================================
# 11. Confirmation
# ============================================================

print(
    "\n================================"
)

print(
    "FIGURE 4 COMPLETE"
)

print(
    "================================"
)


print(
    "\nPNG:"
)

print(
    png_path
)


print(
    "\nPDF:"
)

print(
    pdf_path
)


print(
    "\nSource data:"
)

print(
    source_path
)


print(
    "\nFormal statistics:"
)

display(
    driver_results[
        driver_results[
            "relationship"
        ].isin(
            relations
        )
    ]
)

In [ ]:
# ==========================================
# Audit extreme values in Figure 4
# ==========================================

for rel in [
    "MAC_to_FIB",
    "MAC_to_FibroticMC"
]:

    print("\n==========================")
    print(rel)
    print("==========================")

    temp = driver_df[
        driver_df["Disease"].isin(
            ["SLE", "GBM"]
        )
    ][
        ["Disease", "Patient", "PC1", rel]
    ].dropna()

    print("\nOverall:")
    print(
        temp[rel].describe(
            percentiles=[
                0.01,
                0.05,
                0.10,
                0.50,
                0.90,
                0.95,
                0.99
            ]
        )
    )

    print("\nValues < -5 by disease:")
    print(
        temp.assign(
            extreme=temp[rel] < -5
        )
        .groupby(
            "Disease",
            observed=True
        )["extreme"]
        .agg(["sum", "count", "mean"])
    )

    print("\nValues < -10 by disease:")
    print(
        temp.assign(
            extreme=temp[rel] < -10
        )
        .groupby(
            "Disease",
            observed=True
        )["extreme"]
        .agg(["sum", "count", "mean"])
    )

In [ ]:
# ============================================================
# ROBUSTNESS: finite-count smoothed neighbor enrichment
# Recalculate MAC/Mono × FIB/Fibrotic MC
# ============================================================

from pathlib import Path

import anndata as ad
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

roi_cells_meta = pd.read_pickle(
    base / "roi782_cell_metadata.pkl"
)

roi_ad = ad.read_h5ad(
    base / "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 1. Common PC1 support
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(["min", "max"])
)

pc_low = pc_ranges["min"].max()
pc_high = pc_ranges["max"].min()

roi_common = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["ANCA", "SLE", "GBM"]
    )
    &
    roi_ad.obs["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()

common_ids = set(
    roi_common.index.astype(str)
)

spatial_cells = roi_cells_meta[
    roi_cells_meta["roi_id"]
    .astype(str)
    .isin(common_ids)
].copy()

print(
    "Common-support cells:",
    len(spatial_cells)
)


# ============================================================
# 2. NEW enrichment function
#
# Important:
# Uses neighbor COUNTS + 0.5 pseudocount
# No arbitrary 1e-6 floor
# ============================================================

def neighbor_enrichment_smoothed(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3,
    alpha=0.5
):

    labels = (
        d["celltype_l1"]
        .astype(str)
        .to_numpy()
    )

    src = labels == source
    tgt = labels == target

    n = len(d)
    n_src = int(src.sum())
    n_tgt = int(tgt.sum())

    if (
        n < 5
        or n_src < min_source
        or n_tgt < min_target
    ):
        return np.nan

    xy = d[
        ["x_centroid", "y_centroid"]
    ].to_numpy()

    tree = cKDTree(xy)

    kk = min(
        k + 1,
        n
    )

    _, nbr = tree.query(
        xy[src],
        k=kk
    )

    if nbr.ndim == 1:
        nbr = nbr[:, None]

    # remove source cell itself
    nbr = nbr[:, 1:]

    if nbr.shape[1] == 0:
        return np.nan

    # --------------------------
    # Actual number of target neighbors
    # --------------------------

    target_hits = int(
        tgt[nbr].sum()
    )

    total_neighbor_slots = int(
        nbr.size
    )

    # --------------------------
    # Smoothed observed probability
    # --------------------------

    p_obs = (
        target_hits + alpha
    ) / (
        total_neighbor_slots
        + 2 * alpha
    )

    # --------------------------
    # Smoothed ROI baseline probability
    # --------------------------

    p_exp = (
        n_tgt + alpha
    ) / (
        (n - 1)
        + 2 * alpha
    )

    return np.log2(
        p_obs / p_exp
    )


# ============================================================
# 3. Four pre-specified relationships
# ============================================================

relations = [
    (
        "MAC",
        "FIB",
        "MAC_to_FIB"
    ),
    (
        "MAC",
        "Fibrotic MC",
        "MAC_to_FibroticMC"
    ),
    (
        "Mono",
        "FIB",
        "Mono_to_FIB"
    ),
    (
        "Mono",
        "Fibrotic MC",
        "Mono_to_FibroticMC"
    )
]


# ============================================================
# 4. Recalculate all ROIs
# ============================================================

rows = []

for roi_id, d in spatial_cells.groupby(
    "roi_id",
    observed=True
):

    row = {
        "roi_id": str(roi_id)
    }

    for source, target, name in relations:

        row[name] = (
            neighbor_enrichment_smoothed(
                d,
                source=source,
                target=target,
                k=6,
                alpha=0.5
            )
        )

    rows.append(row)


smooth_df = (
    pd.DataFrame(rows)
    .set_index("roi_id")
)


# ============================================================
# 5. Add metadata
# ============================================================

smooth_df["Disease"] = (
    roi_ad.obs.loc[
        smooth_df.index,
        "Disease"
    ].astype(str)
)

smooth_df["Patient"] = (
    roi_ad.obs.loc[
        smooth_df.index,
        "Patient_Sample_ID"
    ].astype(str)
)

smooth_df["PC1"] = (
    roi_ad.obs.loc[
        smooth_df.index,
        "PC1_crescent"
    ]
)


driver_vars = [
    "MAC_to_FIB",
    "MAC_to_FibroticMC",
    "Mono_to_FIB",
    "Mono_to_FibroticMC"
]


# ============================================================
# 6. Formal LN vs anti-GBM tests
# ============================================================

result_rows = []

for var in driver_vars:

    d = smooth_df[
        smooth_df["Disease"].isin(
            ["SLE", "GBM"]
        )
    ].dropna(
        subset=[var]
    ).copy()

    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=["SLE", "GBM"]
    )

    # patient-balanced
    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = (
        1.0 / n_roi
    )

    fit = smf.wls(
        (
            f"{var} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    terms = [
        term
        for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(terms):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    result_rows.append([
        var,
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1]
    ])


smooth_results = pd.DataFrame(
    result_rows,
    columns=[
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns"
    ]
)

smooth_results["FDR"] = multipletests(
    smooth_results["pvalue"],
    method="fdr_bh"
)[1]

smooth_results = (
    smooth_results
    .sort_values("FDR")
)


print(
    "\n=============================="
)

print(
    "SMOOTHED LN vs anti-GBM RESULTS"
)

print(
    "=============================="
)

display(
    smooth_results
)


# ============================================================
# 7. Check distributions after correction
# ============================================================

print(
    "\nMAC → Fibrotic MC corrected distribution:"
)

temp = smooth_df[
    smooth_df["Disease"].isin(
        ["SLE", "GBM"]
    )
][
    [
        "Disease",
        "MAC_to_FibroticMC"
    ]
].dropna()

print(
    temp[
        "MAC_to_FibroticMC"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "\nBy disease:"
)

print(
    temp.groupby(
        "Disease",
        observed=True
    )[
        "MAC_to_FibroticMC"
    ].describe()
)


# ============================================================
# 8. Save
# ============================================================

smooth_df.to_csv(
    base /
    "figure4_driver_neighbor_data_smoothed.csv"
)

smooth_results.to_csv(
    base /
    "figure4_driver_results_smoothed.csv",
    index=False
)

print(
    "\nSaved corrected results."
)

In [ ]:
# ============================================================
# FIGURE 4 ROBUSTNESS
# Smoothed k sensitivity + anti-GBM leave-one-out
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from scipy.spatial import cKDTree
from patsy import bs
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. Relationships to validate
# ============================================================

target_relations = [
    (
        "MAC",
        "FIB",
        "MAC_to_FIB"
    ),
    (
        "MAC",
        "Fibrotic MC",
        "MAC_to_FibroticMC"
    )
]


# ============================================================
# 2. Re-use finite-count smoothed enrichment
# ============================================================

def neighbor_enrichment_smoothed(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3,
    alpha=0.5
):

    labels = (
        d["celltype_l1"]
        .astype(str)
        .to_numpy()
    )

    src = labels == source
    tgt = labels == target

    n = len(d)
    n_src = int(src.sum())
    n_tgt = int(tgt.sum())

    if (
        n < 5
        or n_src < min_source
        or n_tgt < min_target
    ):
        return np.nan

    xy = d[
        ["x_centroid", "y_centroid"]
    ].to_numpy()

    tree = cKDTree(xy)

    kk = min(k + 1, n)

    _, nbr = tree.query(
        xy[src],
        k=kk
    )

    if nbr.ndim == 1:
        nbr = nbr[:, None]

    nbr = nbr[:, 1:]

    if nbr.shape[1] == 0:
        return np.nan

    target_hits = int(
        tgt[nbr].sum()
    )

    total_slots = int(
        nbr.size
    )

    p_obs = (
        target_hits + alpha
    ) / (
        total_slots + 2 * alpha
    )

    p_exp = (
        n_tgt + alpha
    ) / (
        (n - 1) + 2 * alpha
    )

    return np.log2(
        p_obs / p_exp
    )


# ============================================================
# 3. Helper for SLE vs GBM spline interaction
# ============================================================

def interaction_pvalue(
    df,
    outcome
):

    d = df[
        df["Disease"].isin(
            ["SLE", "GBM"]
        )
    ].dropna(
        subset=[outcome]
    ).copy()

    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=["SLE", "GBM"]
    )

    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = (
        1.0 / n_roi
    )

    fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    terms = [
        term
        for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(terms):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    return (
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1]
    )


# ============================================================
# 4. k sensitivity
# ============================================================

k_values = [
    4,
    6,
    8,
    10
]

k_rows = []


for k in k_values:

    rows = []

    for roi_id, d in spatial_cells.groupby(
        "roi_id",
        observed=True
    ):

        row = {
            "roi_id": str(roi_id)
        }

        for source, target, name in target_relations:

            row[name] = (
                neighbor_enrichment_smoothed(
                    d,
                    source=source,
                    target=target,
                    k=k,
                    alpha=0.5
                )
            )

        rows.append(row)


    temp = (
        pd.DataFrame(rows)
        .set_index("roi_id")
    )


    temp["Disease"] = (
        roi_ad.obs.loc[
            temp.index,
            "Disease"
        ].astype(str)
    )

    temp["Patient"] = (
        roi_ad.obs.loc[
            temp.index,
            "Patient_Sample_ID"
        ].astype(str)
    )

    temp["PC1"] = (
        roi_ad.obs.loc[
            temp.index,
            "PC1_crescent"
        ]
    )


    for _, _, name in target_relations:

        p, rank, ncols = (
            interaction_pvalue(
                temp,
                name
            )
        )

        k_rows.append([
            k,
            name,
            p,
            rank,
            ncols
        ])


k_results = pd.DataFrame(
    k_rows,
    columns=[
        "k",
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns"
    ]
)


print(
    "\n=============================="
)

print(
    "SMOOTHED k SENSITIVITY"
)

print(
    "=============================="
)

display(
    k_results
)


# ============================================================
# 5. GBM leave-one-out at prespecified k=6
# ============================================================

gbm_patients = (
    smooth_df.loc[
        smooth_df["Disease"] == "GBM",
        "Patient"
    ]
    .astype(str)
    .unique()
)


loo_rows = []


for dropped in gbm_patients:

    for relation in [
        "MAC_to_FIB",
        "MAC_to_FibroticMC"
    ]:

        d = smooth_df[
            smooth_df["Disease"].isin(
                ["SLE", "GBM"]
            )
        ].dropna(
            subset=[relation]
        ).copy()

        d = d[
            d["Patient"].astype(str)
            != str(dropped)
        ].copy()

        p, rank, ncols = (
            interaction_pvalue(
                d,
                relation
            )
        )

        loo_rows.append([
            dropped,
            relation,
            p,
            rank,
            ncols
        ])


loo_results = pd.DataFrame(
    loo_rows,
    columns=[
        "dropped_GBM_patient",
        "relationship",
        "pvalue",
        "matrix_rank",
        "n_columns"
    ]
)


print(
    "\n=============================="
)

print(
    "SMOOTHED GBM LEAVE-ONE-OUT"
)

print(
    "=============================="
)

display(
    loo_results
)


# ============================================================
# 6. Stability summary
# ============================================================

loo_summary = (
    loo_results
    .groupby(
        "relationship",
        observed=True
    )["pvalue"]
    .agg(
        median_p="median",
        max_p="max",
        min_p="min"
    )
)


loo_stability = (
    loo_results
    .assign(
        p_lt_005=
        loo_results["pvalue"] < 0.05
    )
    .groupby(
        "relationship",
        observed=True
    )["p_lt_005"]
    .mean()
)


print(
    "\nLOO summary:"
)

display(
    loo_summary
)


print(
    "\nFraction P < 0.05:"
)

display(
    loo_stability
)


# ============================================================
# 7. Save
# ============================================================

k_results.to_csv(
    base /
    "figure4_smoothed_k_sensitivity.csv",
    index=False
)

loo_results.to_csv(
    base /
    "figure4_smoothed_GBM_LOO.csv",
    index=False
)

print(
    "\nSaved robustness results."
)

In [ ]:
# ============================================================
# FIGURE 4 — COMPLETE FINAL VERSION
#
# A MAC → Fibrotic MC trajectory
# B MAC → FIB trajectory
# C k sensitivity
# D anti-GBM leave-one-out
#
# Completely self-contained
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Read corrected/smoothed results
# ============================================================

smooth_df = pd.read_csv(
    base / "figure4_driver_neighbor_data_smoothed.csv",
    index_col=0
)

smooth_results = pd.read_csv(
    base / "figure4_driver_results_smoothed.csv"
)

k_results = pd.read_csv(
    base / "figure4_smoothed_k_sensitivity.csv"
)

loo_results = pd.read_csv(
    base / "figure4_smoothed_GBM_LOO.csv"
)


print("Smoothed data:", smooth_df.shape)

print("\nFormal results:")
display(smooth_results)


# ============================================================
# 2. Formal display settings
# ============================================================

palette = {
    "SLE": "#E69F00",
    "GBM": "#009E73"
}

display_name = {
    "SLE": "LN",
    "GBM": "anti-GBM"
}


# ============================================================
# 3. Keep LN vs anti-GBM
# ============================================================

plot_df = smooth_df[
    smooth_df["Disease"].isin(
        ["SLE", "GBM"]
    )
].copy()


# ============================================================
# 4. Formal model + predictions
# ============================================================

def fit_prediction(
    outcome
):

    d = plot_df.dropna(
        subset=[
            outcome,
            "PC1",
            "Disease",
            "Patient"
        ]
    ).copy()


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform(
        "size"
    )


    d["patient_weight"] = (
        1.0 / n_roi
    )


    fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )


    grid = np.linspace(
        d["PC1"].min(),
        d["PC1"].max(),
        180
    )


    pred_list = []


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1": grid,

            "Disease": pd.Categorical(
                [disease] * len(grid),
                categories=[
                    "SLE",
                    "GBM"
                ]
            )
        })


        pred = fit.get_prediction(
            newdata
        ).summary_frame(
            alpha=0.05
        )


        temp = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                disease,

            "fit":
                pred["mean"].to_numpy(),

            "lower":
                pred["mean_ci_lower"].to_numpy(),

            "upper":
                pred["mean_ci_upper"].to_numpy()
        })


        pred_list.append(
            temp
        )


    return (
        d,
        pd.concat(
            pred_list,
            ignore_index=True
        )
    )


# ============================================================
# 5. Helper for FDR
# ============================================================

def get_fdr(
    relationship
):

    return float(
        smooth_results.loc[
            smooth_results["relationship"]
            == relationship,
            "FDR"
        ].iloc[0]
    )


def format_fdr(
    q
):

    if q < 0.001:
        return f"{q:.2e}"

    return f"{q:.3f}"


# ============================================================
# 6. Create 2 × 2 Figure
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.5, 10)
)

axA, axB = axes[0]
axC, axD = axes[1]


# ============================================================
# Helper for trajectory panels
# ============================================================

def trajectory_panel(
    ax,
    outcome,
    title,
    panel
):

    d, pred_df = fit_prediction(
        outcome
    )


    # Only clip scatter display,
    # NOT the statistical model
    q01 = d[outcome].quantile(
        0.01
    )

    q99 = d[outcome].quantile(
        0.99
    )


    d["display_y"] = (
        d[outcome]
        .clip(
            lower=q01,
            upper=q99
        )
    )


    for disease in [
        "SLE",
        "GBM"
    ]:

        raw = d[
            d["Disease"].astype(str)
            == disease
        ]


        pred = pred_df[
            pred_df["Disease"]
            == disease
        ]


        color = palette[
            disease
        ]


        ax.scatter(
            raw["PC1"],
            raw["display_y"],
            s=18,
            alpha=0.20,
            color=color
        )


        ax.plot(
            pred["PC1"],
            pred["fit"],
            linewidth=2.6,
            color=color,
            label=display_name[
                disease
            ]
        )


        ax.fill_between(
            pred["PC1"],
            pred["lower"],
            pred["upper"],
            color=color,
            alpha=0.14
        )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="#4C78A8"
    )


    q = get_fdr(
        outcome
    )


    ax.set_title(
        (
            f"{panel}  {title}\n"
            "LN vs anti-GBM interaction "
            f"FDR = {format_fdr(q)}"
        ),
        loc="left",
        fontsize=12
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )

    ax.set_ylabel(
        "log2 neighbor enrichment"
    )


    ax.legend(
        frameon=False
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ============================================================
# A. MAC → Fibrotic MC
# ============================================================

trajectory_panel(
    axA,
    "MAC_to_FibroticMC",
    "MAC → Fibrotic MC",
    "A"
)


# ============================================================
# B. MAC → FIB
# ============================================================

trajectory_panel(
    axB,
    "MAC_to_FIB",
    "MAC → FIB",
    "B"
)


# ============================================================
# C. k sensitivity
# ============================================================

ax = axC


for relationship, label in [
    (
        "MAC_to_FibroticMC",
        "MAC → Fibrotic MC"
    ),
    (
        "MAC_to_FIB",
        "MAC → FIB"
    )
]:

    d = (
        k_results[
            k_results["relationship"]
            == relationship
        ]
        .sort_values("k")
    )


    ax.plot(
        d["k"],
        d["pvalue"],
        marker="o",
        linewidth=2,
        label=label
    )


ax.axhline(
    0.05,
    linestyle="--",
    linewidth=1
)

ax.set_yscale(
    "log"
)

ax.set_xticks(
    [4, 6, 8, 10]
)

ax.set_xlabel(
    "Nearest neighbors (k)"
)

ax.set_ylabel(
    "Interaction P value"
)

ax.set_title(
    "C  Neighborhood-scale sensitivity",
    loc="left",
    fontsize=12
)

ax.legend(
    frameon=False
)

ax.text(
    0.96,
    0.05,
    "Both relations: 4/4 P < 0.05",
    transform=ax.transAxes,
    ha="right",
    va="bottom"
)


# ============================================================
# D. anti-GBM leave-one-out
# ============================================================

ax = axD


for relationship, label, marker in [
    (
        "MAC_to_FibroticMC",
        "MAC → Fibrotic MC",
        "o"
    ),
    (
        "MAC_to_FIB",
        "MAC → FIB",
        "s"
    )
]:

    d = (
        loo_results[
            loo_results["relationship"]
            == relationship
        ]
        .copy()
    )


    # 保证患者顺序固定
    d = d.sort_values(
        "dropped_GBM_patient"
    )


    x = np.arange(
        len(d)
    )


    ax.plot(
        x,
        d["pvalue"],
        marker=marker,
        linewidth=1.8,
        label=label
    )


ax.axhline(
    0.05,
    linestyle="--",
    linewidth=1
)

ax.set_yscale(
    "log"
)


# 用一个relationship拿患者顺序
patients = (
    loo_results[
        loo_results["relationship"]
        == "MAC_to_FIB"
    ]
    .sort_values(
        "dropped_GBM_patient"
    )[
        "dropped_GBM_patient"
    ]
    .tolist()
)


ax.set_xticks(
    np.arange(
        len(patients)
    )
)

ax.set_xticklabels(
    patients,
    rotation=45,
    ha="right"
)

ax.set_xlabel(
    "anti-GBM patient excluded"
)

ax.set_ylabel(
    "Interaction P value"
)

ax.set_title(
    "D  Leave-one-anti-GBM-patient-out",
    loc="left",
    fontsize=12
)

ax.legend(
    frameon=False
)

ax.text(
    0.96,
    0.05,
    "Both relations: 5/5 P < 0.05",
    transform=ax.transAxes,
    ha="right",
    va="bottom"
)


# ============================================================
# 7. Final layout
# ============================================================

plt.tight_layout(
    w_pad=3,
    h_pad=3
)


# ============================================================
# 8. Save
# ============================================================

png_path = (
    figdir /
    "Figure4_COMPLETE_macrophage_stromal_drivers.png"
)

pdf_path = (
    figdir /
    "Figure4_COMPLETE_macrophage_stromal_drivers.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


print(
    "\nFigure 4 saved:"
)

print(
    png_path
)

print(
    pdf_path
)

In [ ]:
# ============================================================
# FINAL FIGURE 4 MODEL-SPECIFICATION ROBUSTNESS
#
# Test:
#   spline df = 3 / 4 / 5
#   PC1 trimming = 0 / 2.5% / 5%
#
# Relations:
#   MAC → Fibrotic MC
#   MAC → FIB
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Load corrected data
# ============================================================

base = Path(str(PROJECT_DIR))

df = pd.read_csv(
    base / "figure4_driver_neighbor_data_smoothed.csv",
    index_col=0
)

df = df[
    df["Disease"].isin(
        ["SLE", "GBM"]
    )
].copy()


relations = [
    "MAC_to_FibroticMC",
    "MAC_to_FIB"
]

spline_dfs = [
    3,
    4,
    5
]

trim_levels = [
    0.00,
    0.025,
    0.05
]


# ============================================================
# 1. Function for joint interaction test
# ============================================================

def run_interaction_test(
    data,
    outcome,
    spline_df
):

    d = data.dropna(
        subset=[
            outcome,
            "PC1",
            "Disease",
            "Patient"
        ]
    ).copy()

    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )

    # Patient-balanced weights
    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = (
        1.0 / n_roi
    )

    fit = smf.wls(
        (
            f"{outcome} ~ "
            f"bs(PC1, df={spline_df}, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    interaction_terms = [
        term
        for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(interaction_terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    return (
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1],
        len(d),
        d["Patient"].nunique(),
        d.loc[
            d["Disease"].astype(str) == "GBM",
            "Patient"
        ].nunique()
    )


# ============================================================
# 2. Run all specifications
# ============================================================

rows = []

for trim in trim_levels:

    # --------------------------------
    # Define shared PC1 range
    #
    # trim=0:
    #   common min/max between LN and GBM
    #
    # trim=.025/.05:
    #   remove tails within each disease,
    #   then take their common overlap
    # --------------------------------

    disease_ranges = {}

    for disease in [
        "SLE",
        "GBM"
    ]:

        x = df.loc[
            df["Disease"] == disease,
            "PC1"
        ].dropna()

        if trim == 0:

            low = x.min()
            high = x.max()

        else:

            low = x.quantile(trim)
            high = x.quantile(
                1 - trim
            )

        disease_ranges[
            disease
        ] = (
            low,
            high
        )

    common_low = max(
        disease_ranges["SLE"][0],
        disease_ranges["GBM"][0]
    )

    common_high = min(
        disease_ranges["SLE"][1],
        disease_ranges["GBM"][1]
    )

    d_trim = df[
        df["PC1"].between(
            common_low,
            common_high
        )
    ].copy()

    for spline_df in spline_dfs:

        for relation in relations:

            (
                p,
                rank,
                ncols,
                n_roi,
                n_patients,
                n_gbm
            ) = run_interaction_test(
                d_trim,
                relation,
                spline_df
            )

            rows.append([
                relation,
                trim,
                spline_df,
                common_low,
                common_high,
                p,
                rank,
                ncols,
                n_roi,
                n_patients,
                n_gbm
            ])


robust_spec = pd.DataFrame(
    rows,
    columns=[
        "relationship",
        "PC1_trim",
        "spline_df",
        "PC1_low",
        "PC1_high",
        "pvalue",
        "matrix_rank",
        "n_columns",
        "n_ROI",
        "n_patients",
        "n_GBM_patients"
    ]
)

robust_spec["p_lt_005"] = (
    robust_spec["pvalue"] < 0.05
)


# ============================================================
# 3. Show full table
# ============================================================

print(
    "===================================="
)

print(
    "MODEL-SPECIFICATION ROBUSTNESS"
)

print(
    "===================================="
)

display(
    robust_spec.sort_values(
        [
            "relationship",
            "PC1_trim",
            "spline_df"
        ]
    )
)


# ============================================================
# 4. Summary
# ============================================================

summary = (
    robust_spec
    .groupby(
        "relationship",
        observed=True
    )
    .agg(
        tests=(
            "pvalue",
            "size"
        ),
        significant_tests=(
            "p_lt_005",
            "sum"
        ),
        min_p=(
            "pvalue",
            "min"
        ),
        median_p=(
            "pvalue",
            "median"
        ),
        max_p=(
            "pvalue",
            "max"
        )
    )
)

summary[
    "fraction_significant"
] = (
    summary[
        "significant_tests"
    ]
    /
    summary[
        "tests"
    ]
)

print(
    "\n===================================="
)

print(
    "SUMMARY"
)

print(
    "===================================="
)

display(
    summary
)


# ============================================================
# 5. Rank check
# ============================================================

print(
    "\nNon-full-rank models:"
)

display(
    robust_spec[
        robust_spec["matrix_rank"]
        !=
        robust_spec["n_columns"]
    ]
)


# ============================================================
# 6. Save
# ============================================================

robust_spec.to_csv(
    base /
    "figure4_final_model_specification_robustness.csv",
    index=False
)

print(
    "\nSaved."
)

In [ ]:
fibmc_spec = (
    robust_spec[
        robust_spec["relationship"]
        == "MAC_to_FibroticMC"
    ][
        [
            "PC1_trim",
            "spline_df",
            "PC1_low",
            "PC1_high",
            "pvalue",
            "n_ROI",
            "n_patients",
            "n_GBM_patients"
        ]
    ]
    .sort_values(
        ["PC1_trim", "spline_df"]
    )
)

display(fibmc_spec)

In [ ]:
fib_spec = (
    robust_spec[
        robust_spec["relationship"]
        == "MAC_to_FIB"
    ][
        [
            "PC1_trim",
            "spline_df",
            "pvalue",
            "n_ROI",
            "n_patients",
            "n_GBM_patients"
        ]
    ]
    .sort_values(
        ["PC1_trim", "spline_df"]
    )
)

display(fib_spec)